# 転置索引を自作してブール検索・フレーズ検索をする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_search/inverted_index.ipynb)

NLP クックブック 20-01 ／ 検索

検索エンジンの基本である転置索引を自分で作り、AND / OR / NOT や語順どおりのフレーズで検索したい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
from collections import defaultdict
from fugashi import Tagger

tagger = Tagger()
docs = [
    "東京都立大学で自然言語処理の研究をしている。",
    "自然言語処理では機械翻訳の研究が盛んだ。",
    "東京で機械学習の勉強会が開かれた。",
    "京都大学は言語処理の研究で知られる。",
    "機械翻訳は自然な言語を処理する技術の一つである。",
    "東京と京都を結ぶ新幹線に乗った。",
]

def tokenize(text):
    return [w.surface for w in tagger(text) if w.feature.pos1 not in ("助詞", "助動詞", "補助記号")]

# 転置索引: 単語 → {文書番号: [出現位置, ...]}
index = defaultdict(dict)
for doc_id, doc in enumerate(docs):
    for pos, term in enumerate(tokenize(doc)):
        index[term].setdefault(doc_id, []).append(pos)

print(len(index), "語")
for term in ["東京", "研究", "処理"]:
    print(term, dict(index[term]))

27 語
東京 {0: [0], 2: [0], 5: [0]}
研究 {0: [6], 1: [5], 3: [4]}
処理 {0: [5], 1: [2], 3: [3], 4: [4]}


In [3]:
ALL = set(range(len(docs)))
posting = lambda term: set(index.get(term, {}))

def AND(*terms): return set.intersection(*[posting(t) for t in terms])
def OR(*terms): return set.union(*[posting(t) for t in terms])
def NOT(term): return ALL - posting(term)

def phrase(*terms):
    """語がこの順番で隣り合って出現する文書 (位置情報を使う)"""
    hits = set()
    for doc_id in AND(*terms):
        if any(all(p + i in index[t][doc_id] for i, t in enumerate(terms)) for p in index[terms[0]][doc_id]):
            hits.add(doc_id)
    return hits

queries = {
    "東京 AND 研究": AND("東京", "研究"),
    "機械 AND (翻訳 OR 学習)": posting("機械") & OR("翻訳", "学習"),
    "研究 AND NOT 東京": posting("研究") & NOT("東京"),
    'フレーズ "自然 言語 処理"': phrase("自然", "言語", "処理"),
    "自然 AND 言語 AND 処理": AND("自然", "言語", "処理"),
}
for q, result in queries.items():
    print(f"{q}\n   → {sorted(result)} {[docs[i][:18] for i in sorted(result)]}")

東京 AND 研究
   → [0] ['東京都立大学で自然言語処理の研究をし']
機械 AND (翻訳 OR 学習)
   → [1, 2, 4] ['自然言語処理では機械翻訳の研究が盛ん', '東京で機械学習の勉強会が開かれた。', '機械翻訳は自然な言語を処理する技術の']
研究 AND NOT 東京
   → [1, 3] ['自然言語処理では機械翻訳の研究が盛ん', '京都大学は言語処理の研究で知られる。']
フレーズ "自然 言語 処理"
   → [0, 1, 4] ['東京都立大学で自然言語処理の研究をし', '自然言語処理では機械翻訳の研究が盛ん', '機械翻訳は自然な言語を処理する技術の']
自然 AND 言語 AND 処理
   → [0, 1, 4] ['東京都立大学で自然言語処理の研究をし', '自然言語処理では機械翻訳の研究が盛ん', '機械翻訳は自然な言語を処理する技術の']


- 転置索引は「この単語を含む文書はどれか」を即座に引ける構造。全文書を走査せずに済むので、数億件の文書でも高速に検索できる。
- 出現位置まで持っておくと、フレーズ検索や「近くに出現する」という条件の検索ができる。最後の 2 つの結果の違い (語順どおりに並んでいるか) がその効果。
- 日本語では単語の区切り方が検索結果を左右する。「東京都立大学」が 1 語になる辞書だと「東京」で検索しても当たらない。対策は、この章の文字 n-gram による検索。
- 実用の検索エンジン (Lucene、Elasticsearch、Tantivy) も、この構造を圧縮し、スコア計算と組み合わせたもの。